# Graph-Enhanced HyDE on SciFact

**Pipeline:** Extract ER → Graph search (LanceDB) → Generate HyDE → BGE + FAISS retrieval

**Requires:** `GRAPHRAG_API_KEY` in `.env`, `output/` directory with graph artifacts

In [1]:
import os, json, sys
from pathlib import Path

# OpenMP fix — must be before any library import
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"

NOTEBOOK_DIR = Path(".").resolve()
REPO_ROOT    = NOTEBOOK_DIR.parent
OUTPUT_DIR   = NOTEBOOK_DIR / "output"
LANCEDB_URI  = str(OUTPUT_DIR / "lancedb")
RESULTS_DIR  = NOTEBOOK_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)
sys.path.insert(0, str(REPO_ROOT / "hydeOnSciFact"))

from dotenv import load_dotenv
load_dotenv(NOTEBOOK_DIR / ".env")
API_KEY = os.environ["GRAPHRAG_API_KEY"]
print("Ready:", NOTEBOOK_DIR)


Ready: /Users/winstondong/CMUCourses/AdvancedNLP/Assignment_3&4/baseline_reproduction/ANLP-HW34/graphrag_hyde


In [2]:
import pandas as pd, lancedb
from openai import OpenAI
from load_data import load_scifact_data

# SciFact
corpus, queries, qrels, _ = load_scifact_data(split="test")
query_ids      = list(queries.keys())
corpus_doc_ids = list(corpus.keys())
corpus_texts   = [corpus[did] for did in corpus_doc_ids]
print(f"Corpus: {len(corpus_texts):,} docs | Queries: {len(query_ids):,}")

# Graph artifacts
entities_df      = pd.read_parquet(OUTPUT_DIR / "entities.parquet")
relationships_df = pd.read_parquet(OUTPUT_DIR / "relationships.parquet")
entity_id_to_title = dict(zip(entities_df["id"],    entities_df["title"]))
entity_desc        = dict(zip(entities_df["title"], entities_df["description"]))
adjacency: dict = {}
for _, row in relationships_df.iterrows():
    for a, b in [(row["source"], row["target"]), (row["target"], row["source"])]:
        adjacency.setdefault(a, []).append((b, row["description"]))
print(f"Entities: {len(entities_df):,} | Relationships: {len(relationships_df):,}")

# LanceDB + OpenAI client
openai_client   = OpenAI(api_key=API_KEY)
EMBED_MODEL_OAI = "text-embedding-3-large"
entity_table    = lancedb.connect(LANCEDB_URI).open_table("entity_description")
print(f"LanceDB: {entity_table.count_rows():,} entities")


/Users/winstondong/miniforge3/envs/adnlp_hyde_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Corpus: 5,183 docs | Queries: 300
Entities: 25,791 | Relationships: 35,372
LanceDB: 25,791 entities


In [6]:
HYDE_PROMPT = """\
You are a biomedical scientist. Write a concise passage (2-4 sentences) that a paper \
supporting or refuting this claim would contain. Use the knowledge graph context.

Claim: {claim}
Extracted entities: {entities}
Extracted relationships: {rels}
Graph entities:
{graph_entities}
Graph relationships:
{graph_rels}

Passage:"""


def extract_er(claim: str) -> dict:
    resp = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content":
            "Extract biomedical entities and relationships from this claim.\n"
            "Return JSON with keys 'entities' (list of strings) and 'relationships' (list of strings).\n\n"
            f"Claim: {claim}\n\nJSON:"}],
        max_tokens=200, temperature=0,
        response_format={"type": "json_object"},
    )
    try:
        r = json.loads(resp.choices[0].message.content)
    except Exception:
        r = {}
    return {"entities": r.get("entities", []), "relationships": r.get("relationships", [])}


def search_graph(er: dict, top_k: int = 3, max_rels: int = 3) -> dict:
    # Flatten and stringify — LLM occasionally returns nested lists or None
    raw = er.get("entities", [])
    entities = [str(e) for e in (raw if isinstance(raw, list) else []) if e and isinstance(e, (str, int, float))]
    if not entities:
        return {"matched_entities": [], "relationships": []}
    vecs = [x.embedding for x in
            openai_client.embeddings.create(model=EMBED_MODEL_OAI, input=entities).data]
    seen, titles = set(), []
    for vec in vecs:
        for _, row in entity_table.search(vec).limit(top_k).to_pandas().iterrows():
            title = entity_id_to_title.get(row.get("id", ""), "")
            if title and title not in seen:
                seen.add(title); titles.append(title)
    seen_r, rels = set(), []
    for t in titles:
        for nbr, desc in adjacency.get(t, [])[:max_rels]:
            r = f"{t} — {desc} — {nbr}"
            if r not in seen_r:
                seen_r.add(r); rels.append(r)
    return {"matched_entities": [{"title": t, "description": entity_desc.get(t, "")} for t in titles],
            "relationships": rels}


def generate_hyde(claim: str, er: dict, ctx: dict) -> str:
    ge = "\n".join(f"  - {e['title']}: {e['description'][:120]}" for e in ctx["matched_entities"][:8]) or "  (none)"
    gr = "\n".join(f"  - {r}" for r in ctx["relationships"][:10]) or "  (none)"
    resp = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": HYDE_PROMPT.format(
            claim=claim,
            entities=", ".join(str(e) for e in er["entities"]) or "(none)",
            rels="; ".join(str(r) for r in er["relationships"]) or "(none)",
            graph_entities=ge, graph_rels=gr,
        )}],
        max_tokens=256, temperature=0.3,
    )
    return (resp.choices[0].message.content or "").strip()

print("Pipeline functions defined.")


Pipeline functions defined.


In [7]:
from tqdm import tqdm

per_query_rows:    list[dict] = []
hypothetical_docs: list[str]  = []

for qid in tqdm(query_ids, desc="Graph-HyDE"):
    claim = queries[qid]
    er    = extract_er(claim)
    ctx   = search_graph(er)
    hyp   = generate_hyde(claim, er, ctx)
    hypothetical_docs.append(hyp)
    per_query_rows.append({
        "query_id":           qid,
        "query":              claim,
        "extracted_entities": er["entities"],
        "extracted_rels":     er["relationships"],
        "graph_entities":     [e["title"] for e in ctx["matched_entities"]],
        "graph_rels":         ctx["relationships"],
        "hypothetical_doc":   hyp,
        "gold_doc_ids":       sorted(qrels[qid]),
    })

intermediate_path = RESULTS_DIR / "graph_hyde_step3_output.jsonl"
with intermediate_path.open("w", encoding="utf-8") as f:
    for row in per_query_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
print(f"Done. Saved {len(per_query_rows)} rows → {intermediate_path}")


Graph-HyDE: 100%|██████████| 300/300 [21:12<00:00,  4.24s/it]

Done. Saved 300 rows → /Users/winstondong/CMUCourses/AdvancedNLP/Assignment_3&4/baseline_reproduction/ANLP-HW34/graphrag_hyde/results/graph_hyde_step3_output.jsonl


In [8]:
import os, json, sys
from pathlib import Path

# OpenMP fix — must be before sentence_transformers import
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"

NOTEBOOK_DIR = Path(".").resolve()
RESULTS_DIR  = NOTEBOOK_DIR / "results"
_p = str(NOTEBOOK_DIR.parent / "hydeOnSciFact")
if _p not in sys.path:
    sys.path.insert(0, _p)

from sentence_transformers import SentenceTransformer
from embed import Embedder
from retrieve import build_faiss_index, retrieve_top_k
from load_data import load_scifact_data
from evaluate import evaluate_run

TOP_K  = 100
TOP_KS = [1, 3, 5, 10, 100]
intermediate_path = RESULTS_DIR / "graph_hyde_step3_output.jsonl"

# Reload from file if kernel crashed
try:
    _ = per_query_rows, hypothetical_docs, query_ids, qrels, corpus_doc_ids
except NameError:
    assert intermediate_path.exists(), f"Run cell-08 first: {intermediate_path}"
    per_query_rows    = [json.loads(l) for l in intermediate_path.open()]
    query_ids         = [r["query_id"]       for r in per_query_rows]
    hypothetical_docs = [r["hypothetical_doc"] for r in per_query_rows]
    corpus, queries, qrels, _ = load_scifact_data(split="test")
    corpus_doc_ids = list(corpus.keys())
    corpus_texts   = [corpus[did] for did in corpus_doc_ids]
    print(f"Reloaded {len(per_query_rows)} rows from file.")

# Build FAISS if missing (fp32 to avoid OpenMP conflict)
try:
    _ = faiss_index, embedder
except NameError:
    try:
        _ = corpus_texts
    except NameError:
        corpus, _, _, _ = load_scifact_data(split="test")
        corpus_doc_ids = list(corpus.keys())
        corpus_texts   = [corpus[did] for did in corpus_doc_ids]
    embedder = Embedder(model_name="BAAI/bge-base-en-v1.5")
    embedder.model = SentenceTransformer("BAAI/bge-base-en-v1.5",
                                         model_kwargs={"torch_dtype": "float32"})
    print("Encoding corpus...")
    faiss_index = build_faiss_index(embedder.encode(corpus_texts))
    print(f"FAISS: {faiss_index.ntotal:,} vectors")

# Retrieve
print("Encoding hypothetical docs...")
_, all_indices = retrieve_top_k(faiss_index, embedder.encode(hypothetical_docs), TOP_K)

per_query_retrieved = {}
for i, qid in enumerate(query_ids):
    docs = [corpus_doc_ids[j] for j in all_indices[i].tolist()]
    per_query_retrieved[qid] = docs
    per_query_rows[i].update({
        "retrieved_doc_ids": docs,
        "hit@1":   int(any(d in qrels[qid] for d in docs[:1])),
        "hit@5":   int(any(d in qrels[qid] for d in docs[:5])),
        "hit@10":  int(any(d in qrels[qid] for d in docs[:10])),
        "hit@100": int(any(d in qrels[qid] for d in docs[:100])),
    })

# Evaluate
metrics = evaluate_run(per_query_retrieved, qrels, TOP_KS)
print("\n=== Graph-HyDE — SciFact ===")
for k in ["Recall@1", "Recall@5", "Recall@10", "Recall@100", "MRR@10", "nDCG@10"]:
    if k in metrics:
        print(f"  {k:<12}: {metrics[k]:.4f}")

# Save
with (RESULTS_DIR / "graph_hyde_metrics.json").open("w") as f:
    json.dump({"metrics": metrics}, f, indent=2)
with (RESULTS_DIR / "graph_hyde_per_query.jsonl").open("w") as f:
    for row in per_query_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
print("Saved to results/")


Encoding corpus...
FAISS: 5,183 vectors
Encoding hypothetical docs...

=== Graph-HyDE — SciFact ===
  Recall@1    : 0.6019
  Recall@5    : 0.8306
  Recall@10   : 0.8951
  Recall@100  : 0.9733
  MRR@10      : 0.7221
  nDCG@10     : 0.7588
Saved to results/
